In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from md_Helpers import paths

# Set any mask to None to keep all values for that variable.
# Examples: selected_temperatures = [0.8], selected_ncells = [30]
selected_temperatures = None
selected_ncells = None
selected_n_timesteps = None

seitz_path = paths.MASTER_CSVS_V3_ROOT / "seitz_master.csv"
seitz = pd.read_csv(seitz_path)

if "n_timesteps" not in seitz.columns and "evolve_nsteps" in seitz.columns:
    seitz["n_timesteps"] = seitz["evolve_nsteps"]

required_columns = [
    "status",
    "Q",
    "Q_uncertainty",
    "rho_liquid",
    "rho_liquid_uncertainty",
    "source_kT",
    "n_cells",
    "n_timesteps",
]
missing = [column for column in required_columns if column not in seitz.columns]
if missing:
    raise RuntimeError(
        "Missing Seitz columns: "
        + ", ".join(missing)
        + ". Re-run Seitz_CSV.ipynb to rebuild seitz_master.csv."
    )

mask = (
    (seitz["status"] == "seitz_computed")
    & seitz["Q"].notna()
    & seitz["rho_liquid"].notna()
    & seitz["source_kT"].notna()
)

if selected_temperatures is not None:
    mask &= seitz["source_kT"].isin(selected_temperatures)
if selected_ncells is not None:
    mask &= seitz["n_cells"].isin(selected_ncells)
if selected_n_timesteps is not None:
    mask &= seitz["n_timesteps"].isin(selected_n_timesteps)

plot_data = (
    seitz[mask]
    .copy()
    .sort_values(["source_kT", "n_cells", "n_timesteps", "rho_liquid"])
)

for column in ["rho_liquid_uncertainty", "Q_uncertainty"]:
    plot_data[column] = pd.to_numeric(plot_data[column], errors="coerce")
    plot_data.loc[plot_data[column] < 0, column] = np.nan

print(f"Plotting {len(plot_data)} rows from {seitz_path}")
display(
    plot_data[[
        "source_kT",
        "n_cells",
        "n_timesteps",
        "rho_liquid",
        "rho_liquid_uncertainty",
        "Q",
        "Q_uncertainty",
    ]]
)

fig, ax = plt.subplots(figsize=(7, 5))

for (kT, n_cells, n_timesteps), group in plot_data.groupby(
    ["source_kT", "n_cells", "n_timesteps"]
):
    xerr = group["rho_liquid_uncertainty"].to_numpy()
    yerr = group["Q_uncertainty"].to_numpy()
    ax.errorbar(
        group["rho_liquid"],
        group["Q"],
        xerr=None if np.isnan(xerr).all() else xerr,
        yerr=None if np.isnan(yerr).all() else yerr,
        marker="o",
        linewidth=1.5,
        capsize=3,
        label=f"kT={kT:g}, Ncells={int(n_cells)}, steps={int(n_timesteps):g}",
    )

ax.set_xlabel(r"$\rho_\mathrm{liquid}$")
ax.set_ylabel("Q")
ax.set_title("Seitz Q vs liquid density")
ax.legend(title="Selection", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()

plt.show()
